<a href="https://colab.research.google.com/github/moise97/Extract_-_Structure_Data_from_SDFs_pharmaceutical_documentation/blob/main/Full_Segmentation_Pipeline_with_Metadata_Output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pdfplumber
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q sentencepiece
!pip install -q pandas

print('✅ All packages installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
✅ All packages installed!


In [7]:
from google.colab import files

print('📂 Upload your multi-document PDF...')
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {pdf_filename}')

📂 Upload your multi-document PDF...


Saving pharma-blob-sample.pdf to pharma-blob-sample (1).pdf

✅ Uploaded: pharma-blob-sample (1).pdf


In [8]:
!pip install pdfplumber -q
import importlib, sys

# Force reload in case pip installed but Python cache is stale
if 'pdfplumber' in sys.modules:
    del sys.modules['pdfplumber']

import pdfplumber
print('✅ pdfplumber ready:', pdfplumber.__version__)

✅ pdfplumber ready: 0.11.9


In [10]:
import pdfplumber

!pip install -q pdfplumber

pages_text = []

with pdfplumber.open(pdf_filename) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        text = text.strip()
        pages_text.append(text)
        preview = text[:80].replace('\n', ' ') if text else '[NO TEXT EXTRACTED]'
        print(f'Page {i:>3}: {preview}...')

print(f'\n✅ Extracted text from {len(pages_text)} pages.')
print(f'   Pages with no text: {sum(1 for t in pages_text if not t)}')

Page   0: Cytiva 100 Results Way Marlborough, MA 01752 United States Page 1 / 1 cytiva.com...
Page   1: Certificate of Quality This product is manufactured in compliance with our ISO 9...
Page   2: Certificate of Quality This product is manufactured in compliance with our ISO 9...
Page   3: Packaging Component Specification Document Number: PKG-SPEC-2023-0847 Revision: ...
Page   4: Packaging Component Specification (continued) Document Number: PKG-SPEC-2023-084...
Page   5: Declaration Regarding Transmissible Spongiform Encephalopathies (BSE/TSE Complia...
Page   6: Material Description Sheet Product: AKTA ready Gradient Flow Section With Inlets...
Page   7: Supplier Qualification Record Supplier Name: Cytiva Sweden AB Supplier Address: ...
Page   8: Supplier Qualification Record (continued) Supplier: Cytiva Sweden AB (SUP-2019-0...
Page   9: Cytiva 100 Results Way Marlborough, MA 01752 United States Global Chain of Custo...

✅ Extracted text from 10 pages.
   Pages with no text: 0


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

print(f'Loading {MODEL_NAME}...')
print('(~2 minutes on first run)\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # no bitsandbytes needed
    device_map='auto'
)
llm_model.eval()

print(f'\n✅ TinyLlama loaded!')
print(f'   Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow — switch to T4 GPU)"}')
print(f'   Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB' if torch.cuda.is_available() else '')

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
(~2 minutes on first run)



`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


✅ TinyLlama loaded!
   Device: Tesla T4
   Memory used: 2.20 GB


In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

print(f'Loading {MODEL_NAME}...')
print('(~2 minutes on first run)\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
llm_model.eval()

print(f'\n✅ TinyLlama loaded!')
if torch.cuda.is_available():
    print(f'   Device : {torch.cuda.get_device_name(0)}')
    print(f'   Memory : {torch.cuda.memory_allocated()/1e9:.2f} GB used')
else:
    print('   ⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
(~2 minutes on first run)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


✅ TinyLlama loaded!
   Device : Tesla T4
   Memory : 4.40 GB used


In [ ]:
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U accelerate
!pip install -q -U transformers

# Force restart the Python runtime so new versions take effect
import os
os.kill(os.getpid(), 9)

In [ ]:
import json
import re

# Valid document types — model must pick from this list
DOC_TYPES = [
    'Cover Letter',
    'Certificate of Quality',
    'Certificate of Conformance',
    'Packing List',
    'Commercial Invoice',
    'Purchase Order',
    'Bill of Lading',
    'Material Safety Data Sheet',
    'Test Report',
    'Delivery Note',
    'Technical Specification',
    'Declaration of Conformity',
    'Other'
]

DOC_TYPES_STR = '\n'.join(f'- {d}' for d in DOC_TYPES)


def ask_tinyllama(prompt: str, max_new_tokens: int = 200) -> str:
    """Send a prompt to TinyLlama and return the response text."""
    # TinyLlama uses ChatML format
    chat = [
        {"role": "system", "content": "You are a precise document analysis assistant. Follow instructions exactly and respond only as instructed."},
        {"role": "user",   "content": prompt}
    ]
    formatted = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(llm_model.device)
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,       # low = deterministic
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decode only the newly generated tokens
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def is_new_document(prev_text: str, curr_text: str) -> bool:
    """
    Returns True if curr_text starts a new document.
    Returns False if it continues the previous document.
    """
    # Truncate to keep prompt within context window
    prev_snippet = prev_text[:600] if prev_text else '[EMPTY PAGE]'
    curr_snippet = curr_text[:600] if curr_text else '[EMPTY PAGE]'

    prompt = f"""You are comparing two consecutive pages from a scanned PDF.

PAGE A (previous page):
\"\"\"
{prev_snippet}
\"\"\"

PAGE B (current page):
\"\"\"
{curr_snippet}
\"\"\"

Does PAGE B start a NEW document, or does it continue the SAME document as PAGE A?

Signs of a NEW document: different title, different company, different document type, starts at page 1, fresh header/logo.
Signs of SAME document: continues mid-sentence, same header/footer, page 2 of N, same document number.

If PAGE B is empty or unclear, assume it continues the same document.

Respond with ONLY a JSON object in this exact format:
{{"is_new_doc": "Yes"}} or {{"is_new_doc": "No"}}

JSON:"""

    raw = ask_tinyllama(prompt, max_new_tokens=60)

    # Parse JSON from response — robust extraction
    try:
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            return str(parsed.get('is_new_doc', 'No')).strip().lower().startswith('y')
    except Exception:
        pass

    # Fallback: look for Yes/No anywhere in the response
    if re.search(r'\byes\b', raw, re.IGNORECASE):
        return True
    return False  # default: assume same document


def classify_document_type(page_text: str) -> str:
    """
    Returns the document type for the given page text.
    Must be one of the values in DOC_TYPES.
    """
    snippet = page_text[:800] if page_text else '[EMPTY PAGE]'

    prompt = f"""You are a document classification assistant.

Read the following page text and identify which document type it belongs to.

PAGE TEXT:
\"\"\"
{snippet}
\"\"\"

You MUST choose exactly one type from this list:
{DOC_TYPES_STR}

If the page is empty or unclear, respond with: Other

Respond with ONLY a JSON object in this exact format:
{{"is_new_doc": "Yes", "doc_type": "<type from list>"}}

JSON:"""

    raw = ask_tinyllama(prompt, max_new_tokens=80)

    # Parse JSON
    try:
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            doc_type = parsed.get('doc_type', '').strip()
            # Validate it's in our allowed list
            for allowed in DOC_TYPES:
                if allowed.lower() in doc_type.lower():
                    return allowed
    except Exception:
        pass

    # Fallback: search raw response for known doc type names
    for allowed in DOC_TYPES:
        if allowed.lower() in raw.lower():
            return allowed

    return 'Other'  # final fallback


print('✅ Functions defined:')
print('   ask_tinyllama(prompt)                    → raw LLM response')
print('   is_new_document(prev_text, curr_text)    → True / False')
print('   classify_document_type(page_text)        → doc type string')

In [ ]:
import time

results      = []
doc_id       = 0
page_in_doc  = 0
current_type = None

total_pages = len(pages_text)
print(f'Classifying {total_pages} pages with TinyLlama...\n')
print('-' * 60)

for page_num in range(total_pages):
    curr_text = pages_text[page_num]

    if page_num == 0:
        # Always a new document
        is_new   = True
        page_in_doc = 0
        print(f'Page {page_num:>3}: First page → classifying...')
        current_type = classify_document_type(curr_text)
        print(f'          doc_id={doc_id} | type="{current_type}" | page_in_doc=0')

    else:
        prev_text = pages_text[page_num - 1]
        is_new    = is_new_document(prev_text, curr_text)

        if is_new:
            doc_id      += 1
            page_in_doc  = 0
            print(f'Page {page_num:>3}: 🆕 NEW doc detected → classifying...')
            current_type = classify_document_type(curr_text)
            print(f'          doc_id={doc_id} | type="{current_type}" | page_in_doc=0')
        else:
            page_in_doc += 1
            print(f'Page {page_num:>3}: Same doc → doc_id={doc_id} | "{current_type}" | page_in_doc={page_in_doc}')

    results.append({
        'page':        page_num,
        'is_new_doc':  'Yes' if is_new else 'No',
        'doc_type':    current_type,
        'page_in_doc': page_in_doc
    })

print('-' * 60)
print(f'\n✅ Done! {total_pages} pages → {doc_id + 1} document(s) found.')

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

print('Classification Results:')
print('=' * 55)
print(df.to_string(index=False))
print('=' * 55)
print(f'Total pages    : {len(df)}')
print(f'Total documents: {df["is_new_doc"].value_counts().get("Yes", 0)}')
print(f'Document types : {df[df["is_new_doc"]=="Yes"]["doc_type"].tolist()}')

In [ ]:
import json
from google.colab import files

json_output = json.dumps(results, indent=2)

print('JSON Output — paste this into your Google Doc:')
print('=' * 55)
print(json_output)
print('=' * 55)

# Save and download
with open('classification_results.json', 'w') as f:
    f.write(json_output)

print('\n✅ Saved. Downloading now...')
files.download('classification_results.json')

In [ ]:
from google.colab import files

df.to_csv('classification_results.csv', index=False)
print('✅ Saved as classification_results.csv')
files.download('classification_results.csv')